In [1]:
import sys
import os
sys.path.insert(0, os.path.abspath('..'))

import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
import warnings
import time
import joblib
import subprocess
warnings.filterwarnings('ignore')

from src.data_utils import load_config, ensure_dir
from src.metrics import evaluate_all

plt.rcParams['font.sans-serif'] = ['Microsoft YaHei', 'SimHei', 'Arial Unicode MS']
plt.rcParams['axes.unicode_minus'] = False

import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader

device = torch.device('cuda')
print(f'PyTorch : {torch.__version__}')
print(f'CUDA    : {torch.cuda.is_available()}')
print(f'GPU     : {torch.cuda.get_device_name(0)}')

PyTorch : 2.5.1+cu121
CUDA    : True
GPU     : NVIDIA GeForce RTX 3090


In [2]:
cfg           = load_config('../configs/config.yaml')
processed_dir = Path('../data/processed')
model_dir     = ensure_dir('../models')
fig_dir       = ensure_dir('../results/figures')
report_dir    = ensure_dir('../results/reports')
LABEL         = '2024_04'

# 加载特征矩阵和C因子标签
features = np.load(processed_dir / f'NSW_test_features_{LABEL}.npy')       # (H, W, 7)
labels   = np.load(processed_dir / f'NSW_test_cfactor_{LABEL}.npy')        # (H, W)

print(f'特征矩阵形状: {features.shape}')
print(f'C因子标签形状: {labels.shape}')
print(f'特征通道    : NDVI, NDWI, BSI, SAVI, B04, B08, B11')
print(f'标签说明    : C因子（0=植被保护，1=裸土）')

# 有效像素
valid_mask = np.isfinite(labels) & np.all(np.isfinite(features), axis=-1)
print(f'\n有效像素数  : {valid_mask.sum():,}')
print(f'有效像素比例: {valid_mask.mean()*100:.1f}%')

X = features[valid_mask]
y = labels[valid_mask]

print(f'\nX 形状: {X.shape}')
print(f'y 形状: {y.shape}')
print(f'y 范围: {y.min():.4f} ~ {y.max():.4f}')
print(f'y 均值: {y.mean():.4f}')
print(f'y 标准差: {y.std():.4f}')

特征矩阵形状: (16811, 18790, 7)
C因子标签形状: (16811, 18790)
特征通道    : NDVI, NDWI, BSI, SAVI, B04, B08, B11
标签说明    : C因子（0=植被保护，1=裸土）

有效像素数  : 315,878,672
有效像素比例: 100.0%

X 形状: (315878672, 7)
y 形状: (315878672,)
y 范围: 0.0001 ~ 0.4498
y 均值: 0.0125
y 标准差: 0.0129


In [3]:
from sklearn.model_selection import train_test_split

N_SAMPLES = 1_000_000
np.random.seed(42)
idx      = np.random.choice(len(X), size=N_SAMPLES, replace=False)
X_sample = X[idx]
y_sample = y[idx]

# 划分 70 / 15 / 15
X_train, X_temp, y_train, y_temp = train_test_split(
    X_sample, y_sample, test_size=0.30, random_state=42)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, random_state=42)

print(f'训练集: {X_train.shape[0]:,} 样本')
print(f'验证集: {X_val.shape[0]:,} 样本')
print(f'测试集: {X_test.shape[0]:,} 样本')
print(f'\ny 范围: {y_train.min():.4f} ~ {y_train.max():.4f}')
print(f'y 均值: {y_train.mean():.4f}')
print(f'注意  : C因子范围0~1，无需对数变换')

训练集: 700,000 样本
验证集: 150,000 样本
测试集: 150,000 样本

y 范围: 0.0001 ~ 0.4498
y 均值: 0.0125
注意  : C因子范围0~1，无需对数变换


In [4]:
from sklearn.ensemble import RandomForestRegressor

print('='*40)
print('模型 1/4 — Random Forest')
print('='*40)

rf = RandomForestRegressor(
    n_estimators=200, max_depth=15,
    min_samples_leaf=4, n_jobs=-1, random_state=42
)

t0 = time.time()
rf.fit(X_train, y_train)
print(f'训练时间: {time.time()-t0:.1f} 秒')

y_pred     = rf.predict(X_val)
y_pred     = np.clip(y_pred, 0, 1)
rf_metrics = evaluate_all(y_val, y_pred, label='Random Forest')

feat_names = ['NDVI','NDWI','BSI','SAVI','B04','B08','B11']
print('\n特征重要性:')
for name, imp in sorted(zip(feat_names, rf.feature_importances_), key=lambda x: -x[1]):
    print(f'  {name:<6}: {imp:.4f}')

joblib.dump(rf, model_dir / 'rf_cfactor_2024_04.joblib')
print(f'\n模型已保存: rf_cfactor_2024_04.joblib')

模型 1/4 — Random Forest
训练时间: 30.3 秒

  评估结果 — Random Forest
  RMSE    : +0.0084
  MAE     : +0.0040
  R2      : +0.5975
  Bias    : -0.0000

特征重要性:
  BSI   : 0.3326
  NDVI  : 0.1969
  B11   : 0.1865
  NDWI  : 0.1126
  B04   : 0.0997
  B08   : 0.0411
  SAVI  : 0.0306

模型已保存: rf_cfactor_2024_04.joblib


In [5]:
import xgboost as xgb

print('='*40)
print('模型 2/4 — XGBoost')
print('='*40)

xgb_model = xgb.XGBRegressor(
    n_estimators=500, max_depth=8,
    learning_rate=0.05, subsample=0.8,
    colsample_bytree=0.8, tree_method='hist',
    device='cuda', random_state=42
)

t0 = time.time()
xgb_model.fit(
    X_train, y_train,
    eval_set=[(X_val, y_val)],
    verbose=100
)
print(f'\n训练时间: {time.time()-t0:.1f} 秒')

y_pred      = np.clip(xgb_model.predict(X_val), 0, 1)
xgb_metrics = evaluate_all(y_val, y_pred, label='XGBoost')

joblib.dump(xgb_model, model_dir / 'xgb_cfactor_2024_04.joblib')
print(f'模型已保存: xgb_cfactor_2024_04.joblib')

模型 2/4 — XGBoost
[0]	validation_0-rmse:0.01288
[100]	validation_0-rmse:0.00866
[200]	validation_0-rmse:0.00869
[300]	validation_0-rmse:0.00874
[400]	validation_0-rmse:0.00881
[499]	validation_0-rmse:0.00886

训练时间: 5.0 秒

  评估结果 — XGBoost
  RMSE    : +0.0088
  MAE     : +0.0040
  R2      : +0.5519
  Bias    : -0.0000
模型已保存: xgb_cfactor_2024_04.joblib


In [8]:
PATCH_SIZE = 256
PATCH_NUM  = 20000

H, W, C   = features.shape
patches_X = []
patches_y = []

np.random.seed(42)
count = 0
while count < PATCH_NUM:
    r = np.random.randint(0, H - PATCH_SIZE)
    c = np.random.randint(0, W - PATCH_SIZE)
    px = features[r:r+PATCH_SIZE, c:c+PATCH_SIZE, :]
    py = labels[r:r+PATCH_SIZE,   c:c+PATCH_SIZE]
    if np.isfinite(py).mean() > 0.9 and np.isfinite(px).all():
        patches_X.append(px)
        patches_y.append(py)
        count += 1

patches_X = np.stack(patches_X).transpose(0,3,1,2).astype('float32')  # (N,7,64,64)
patches_y = np.nan_to_num(np.stack(patches_y), nan=0).astype('float32')  # (N,64,64)

split     = int(len(patches_X) * 0.85)
Xp_train, Xp_val = patches_X[:split], patches_X[split:]
yp_train, yp_val = patches_y[:split], patches_y[split:]

print(f'Patch 数量  : {len(patches_X)}')
print(f'训练 patch  : {len(Xp_train)}   验证 patch: {len(Xp_val)}')
print(f'y 范围      : {patches_y.min():.4f} ~ {patches_y.max():.4f}')

Patch 数量  : 20000
训练 patch  : 17000   验证 patch: 3000
y 范围      : 0.0001 ~ 0.4498


In [9]:
print('='*40)
print('模型 3/4 — 2D CNN（5层+残差）')
print('='*40)

class ResBlock(nn.Module):
    def __init__(self, channels):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(channels, channels, 3, padding=1),
            nn.BatchNorm2d(channels),
            nn.ReLU(),
            nn.Conv2d(channels, channels, 3, padding=1),
            nn.BatchNorm2d(channels),
        )
        self.relu = nn.ReLU()
    def forward(self, x):
        return self.relu(x + self.block(x))

class CNN2D(nn.Module):
    def __init__(self, in_channels=7):
        super().__init__()
        self.encoder = nn.Sequential(
            # 层1
            nn.Conv2d(in_channels, 32, 3, padding=1),
            nn.BatchNorm2d(32), nn.ReLU(),
            # 层2
            nn.Conv2d(32, 64, 3, padding=1),
            nn.BatchNorm2d(64), nn.ReLU(),
            # 残差块1（层3+4）
            ResBlock(64),
            # 层5
            nn.Conv2d(64, 128, 3, padding=1),
            nn.BatchNorm2d(128), nn.ReLU(),
            # 残差块2（层6+7）
            ResBlock(128),
            # 输出层
            nn.Conv2d(128, 1, 1),
            nn.Sigmoid()
        )
    def forward(self, x):
        return self.encoder(x).squeeze(1)

model_cnn = CNN2D().to(device)
total_params = sum(p.numel() for p in model_cnn.parameters())
print(f'模型参数量: {total_params:,}')

opt_cnn   = torch.optim.Adam(model_cnn.parameters(), lr=5e-4, weight_decay=1e-5)
sch_cnn   = torch.optim.lr_scheduler.CosineAnnealingLR(opt_cnn, T_max=100)
criterion = nn.MSELoss()

dl_tr = DataLoader(
    TensorDataset(torch.FloatTensor(Xp_train), torch.FloatTensor(yp_train)),
    batch_size=16, shuffle=True, num_workers=0
)
dl_vl = DataLoader(
    TensorDataset(torch.FloatTensor(Xp_val), torch.FloatTensor(yp_val)),
    batch_size=16, num_workers=0
)

EPOCHS    = 100
best_loss = float('inf')
best_state = None
t0        = time.time()

for epoch in range(EPOCHS):
    model_cnn.train()
    tr_loss = 0
    for xb, yb in dl_tr:
        xb, yb = xb.to(device), yb.to(device)
        opt_cnn.zero_grad()
        loss = criterion(model_cnn(xb), yb)
        loss.backward()
        opt_cnn.step()
        tr_loss += loss.item()
    sch_cnn.step()

    if (epoch+1) % 10 == 0:
        model_cnn.eval()
        vl_loss = 0
        with torch.no_grad():
            for xb, yb in dl_vl:
                vl_loss += criterion(model_cnn(xb.to(device)), yb.to(device)).item()
        vl_loss /= len(dl_vl)
        # 保存最优权重
        if vl_loss < best_loss:
            best_loss  = vl_loss
            best_state = {k: v.clone() for k, v in model_cnn.state_dict().items()}
        print(f'Epoch {epoch+1:3d}/{EPOCHS} | 训练Loss: {tr_loss/len(dl_tr):.6f} | 验证Loss: {vl_loss:.6f} | 最优: {best_loss:.6f}')

print(f'\n训练时间: {time.time()-t0:.1f} 秒')

# 加载最优权重评估
model_cnn.load_state_dict(best_state)
model_cnn.eval()
y_preds, y_trues = [], []
with torch.no_grad():
    for xb, yb in dl_vl:
        y_preds.append(model_cnn(xb.to(device)).cpu().numpy().flatten())
        y_trues.append(yb.numpy().flatten())

cnn_metrics = evaluate_all(np.concatenate(y_trues), np.concatenate(y_preds), label='2D CNN (5层+残差)')

torch.save(best_state, model_dir / 'cnn2d_cfactor_2024_04.pth')
print(f'模型已保存: cnn2d_cfactor_2024_04.pth')

模型 3/4 — 2D CNN（5层+残差）
模型参数量: 464,769
Epoch  10/100 | 训练Loss: 0.000112 | 验证Loss: 0.000145 | 最优: 0.000145
Epoch  20/100 | 训练Loss: 0.000102 | 验证Loss: 0.000116 | 最优: 0.000116
Epoch  30/100 | 训练Loss: 0.000093 | 验证Loss: 0.000117 | 最优: 0.000116
Epoch  40/100 | 训练Loss: 0.000089 | 验证Loss: 0.000116 | 最优: 0.000116
Epoch  50/100 | 训练Loss: 0.000087 | 验证Loss: 0.000120 | 最优: 0.000116
Epoch  60/100 | 训练Loss: 0.000086 | 验证Loss: 0.000113 | 最优: 0.000113
Epoch  70/100 | 训练Loss: 0.000078 | 验证Loss: 0.000108 | 最优: 0.000108
Epoch  80/100 | 训练Loss: 0.000076 | 验证Loss: 0.000120 | 最优: 0.000108
Epoch  90/100 | 训练Loss: 0.000076 | 验证Loss: 0.000112 | 最优: 0.000108
Epoch 100/100 | 训练Loss: 0.000073 | 验证Loss: 0.000112 | 最优: 0.000108

训练时间: 20808.7 秒

  评估结果 — 2D CNN (5层+残差)
  RMSE    : +0.0104
  MAE     : +0.0041
  R2      : +0.4307
  Bias    : -0.0004
模型已保存: cnn2d_cfactor_2024_04.pth


In [ ]:
import segmentation_models_pytorch as smp

print('='*40)
print('模型 4/4 — U-Net')
print('='*40)

unet = smp.Unet(
    encoder_name='resnet34',
    encoder_weights=None,
    in_channels=7,
    classes=1,
    activation='sigmoid'  # C因子范围0~1
).to(device)

total_params = sum(p.numel() for p in unet.parameters())
print(f'模型参数量: {total_params:,}')

opt_u      = torch.optim.Adam(unet.parameters(), lr=1e-4, weight_decay=1e-5)
sch_u      = torch.optim.lr_scheduler.CosineAnnealingLR(opt_u, T_max=100)
criterion  = nn.MSELoss()

dl_tr_u = DataLoader(
    TensorDataset(torch.FloatTensor(Xp_train), torch.FloatTensor(yp_train)),
    batch_size=64, shuffle=True, num_workers=0, pin_memory=True
)
dl_vl_u = DataLoader(
    TensorDataset(torch.FloatTensor(Xp_val), torch.FloatTensor(yp_val)),
    batch_size=64, num_workers=0, pin_memory=True
)

EPOCHS_U   = 100
best_loss  = float('inf')
best_state = None
t0         = time.time()

for epoch in range(EPOCHS_U):
    unet.train()
    tr_loss = 0
    for xb, yb in dl_tr_u:
        xb, yb = xb.to(device), yb.to(device)
        opt_u.zero_grad()
        pred = unet(xb).squeeze(1)
        loss = criterion(pred, yb)
        loss.backward()
        opt_u.step()
        tr_loss += loss.item()
    sch_u.step()

    if (epoch+1) % 10 == 0:
        unet.eval()
        vl_loss = 0
        with torch.no_grad():
            for xb, yb in dl_vl_u:
                vl_loss += criterion(unet(xb.to(device)).squeeze(1), yb.to(device)).item()
        vl_loss /= len(dl_vl_u)
        if vl_loss < best_loss:
            best_loss  = vl_loss
            best_state = {k: v.clone() for k, v in unet.state_dict().items()}
        print(f'Epoch {epoch+1:3d}/{EPOCHS_U} | 训练Loss: {tr_loss/len(dl_tr_u):.6f} | 验证Loss: {vl_loss:.6f} | 最优: {best_loss:.6f}')

print(f'\n训练时间: {time.time()-t0:.1f} 秒')

unet.load_state_dict(best_state)
unet.eval()
y_preds, y_trues = [], []
with torch.no_grad():
    for xb, yb in dl_vl_u:
        y_preds.append(unet(xb.to(device)).squeeze(1).cpu().numpy().flatten())
        y_trues.append(yb.numpy().flatten())

unet_metrics = evaluate_all(
    np.concatenate(y_trues),
    np.concatenate(y_preds),
    label='U-Net'
)

torch.save(best_state, model_dir / 'unet_cfactor_2024_04.pth')
print(f'模型已保存: unet_cfactor_2024_04.pth')

模型 4/4 — U-Net
模型参数量: 24,448,913
Epoch  10/100 | 训练Loss: 0.000261 | 验证Loss: 0.000267 | 最优: 0.000267
